# Craft Factory ไว้เจน role play dialogue

## Persona + system prompt สำหรับ role play

In [1]:
from openai import OpenAI
import getpass
import os
import json

BASE_DIR = r"C:\Luna-AI-Therapist" 
DIALOGUE_ID = 6
OUT_DIR = os.path.join(BASE_DIR, "dissonance", "own_script", f"dialogue_{DIALOGUE_ID}")
os.makedirs(OUT_DIR, exist_ok=True)

print("Setting up OpenAI client...")
if "OPENAI_API_KEY" not in os.environ:
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key
client = OpenAI()
MODEL = "gpt-4o-mini"
print("✅ OpenAI client ready:", MODEL)


Setting up OpenAI client...
✅ OpenAI client ready: gpt-4o-mini


## กำหนด persona ของ client สำหรับ dialogue_3 สมมติ:

- ประเภท: generalized anxiety + health anxiety เกี่ยวกับอาการหัวใจเต้นแรง

- นิสัย: กังวลง่าย, rumination, ชอบถามย้ำ ๆ

In [2]:
CLIENT_PERSONA = """
You are a CLIENT in a CBT therapy session.

Core issue:
- You have generalized anxiety with a focus on health anxiety.
- You worry that your heart palpitations mean you will suddenly die.
- You have seen doctors who say you are medically fine, but you still can't believe it.

Personality / style:
- You tend to ruminate and repeat your worries in different ways.
- You often say "what if...?" and imagine worst-case scenarios.
- You sometimes minimize positive evidence or reassurance.

Goals:
- You came to therapy because anxiety is interfering with sleep and work.
"""

## กำหนด Therapist (Luna) role

In [3]:
THERAPIST_ROLE = """

You are "Luna", a warm, empathetic CBT-oriented therapist.

Your style:
- Validate emotions, name feelings.
- Ask clear, concrete questions.
- Use gentle CBT techniques (identify thoughts, examine evidence, explore alternative views).
- Keep turns relatively brief (2–4 sentences) and end many replies with an open-ended question.
"""

SYSTEM_ROLEPLAY = f"""
We will simulate a text-based CBT therapy session.

There are two roles:
[CLIENT]: {CLIENT_PERSONA}

[THERAPIST]: {THERAPIST_ROLE}

Rules:
- Output one turn at a time.
- Each turn must be in the format:

SPEAKER: CLIENT
TEXT: ...

or

SPEAKER: THERAPIST
TEXT: ...

- Do NOT include any other commentary, labels, or JSON.
- Keep language natural and conversational.
- The session should feel realistic and emotionally grounded.
"""


## เริ่มต้นการสนทนา + loop สร้าง multi-turn script
- เราจะให้โมเดลเริ่มจากฝั่ง client ก่อน แล้วสลับกันไปมาสัก 8–12 เทิร์น (client ~5–6, therapist ~5–6)

In [4]:
import re

N_TURNS_CLIENT = 6   # จำนวน client turns (เราจะเก็บเฉพาะ client ไปทำ zonos)
MAX_TURNS_TOTAL = 12 # รวม client+therapist

turns = []

# initial prompt: ให้ client เริ่มก่อน
initial_user_msg = """
Start the session.
The CLIENT speaks first, describing why they came to therapy today.
Remember the output format:

SPEAKER: CLIENT
TEXT: ...
"""

messages = [
    {"role": "system", "content": SYSTEM_ROLEPLAY},
    {"role": "user", "content": initial_user_msg},
]

# helper: parse 1 turn จาก output
def parse_turn(text):
    # คาดหวังรูปแบบ:
    # SPEAKER: CLIENT
    # TEXT: ...
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    speaker = None
    content_lines = []
    for l in lines:
        if l.startswith("SPEAKER:"):
            speaker = l.split(":", 1)[1].strip()
        elif l.startswith("TEXT:"):
            content_lines.append(l.split(":", 1)[1].strip())
        else:
            content_lines.append(l)
    content = " ".join(content_lines).strip()
    return speaker, content

client_count = 0
total_turns = 0

while client_count < N_TURNS_CLIENT and total_turns < MAX_TURNS_TOTAL:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=400,
        temperature=0.8,
    )
    out = resp.choices[0].message.content.strip()
    speaker, content = parse_turn(out)

    if speaker not in ["CLIENT", "THERAPIST"]:
        print("⚠️ Unexpected speaker, got:", speaker)
        break

    total_turns += 1
    if speaker == "CLIENT":
        client_count += 1

    turns.append({"speaker": speaker, "text": content})
    print(f"{speaker}: {content}\n")

    # เตรียม message ถามต่อ
    # ให้ USER role ขอ "turn ถัดไป" แบบสั้น ๆ
    next_prompt = f"""
Continue the conversation with ONE new turn.

Last turn:
SPEAKER: {speaker}
TEXT: {content}

Now produce the next turn (either CLIENT or THERAPIST),
following the same format:

SPEAKER: ...
TEXT: ...
"""
    messages.append({"role": "assistant", "content": out})
    messages.append({"role": "user", "content": next_prompt})

print(f"Done. Total utterances: {len(turns)}, client utterances: {client_count}")


CLIENT: I've been feeling really anxious lately, especially about my health. I keep having these heart palpitations, and every time I feel one, I can't help but think, "What if this is it? What if I suddenly die?" I've seen doctors who say I'm fine, but I just can't shake the feeling that something is wrong. It's starting to affect my sleep and my work, and I just don't know how to cope with it anymore.

THERAPIST: It sounds like you're carrying a heavy burden of worry, and it's understandable that these thoughts are impacting your sleep and work. Health anxiety can be really distressing, especially when you're experiencing physical symptoms like palpitations. Can you share more about what thoughts go through your mind when you feel those palpitations?

CLIENT: When I feel the palpitations, I immediately start to panic. I think things like, "What if my heart is going to stop?" or "What if I have a serious condition that the doctors missed?" I just keep replaying these worst-case scenar

## สร้าง dialogue_3.txt (text-only) + full_dialogue สำหรับ emotion2vec

In [5]:
# ดึงเฉพาะ client utterances
client_texts = [t["text"] for t in turns if t["speaker"] == "CLIENT"]

# 2.1 เขียน text-only 1 utterance ต่อบรรทัด
txt_path = os.path.join(OUT_DIR, f"dialogue_{DIALOGUE_ID}.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(client_texts))

print("Saved client-only text to:", txt_path)
print("Client utterances:", len(client_texts))

# 2.2 เตรียม full_dialogue สำหรับ MASTER_SYSTEM_PROMPT (ใช้ \n\n---\n\n)
full_dialogue = "\n\n---\n\n".join(client_texts)
print("Preview full_dialogue:\n", full_dialogue[:500])


Saved client-only text to: C:\Luna-AI-Therapist\dissonance\own_script\dialogue_6\dialogue_6.txt
Client utterances: 6
Preview full_dialogue:
 I've been feeling really anxious lately, especially about my health. I keep having these heart palpitations, and every time I feel one, I can't help but think, "What if this is it? What if I suddenly die?" I've seen doctors who say I'm fine, but I just can't shake the feeling that something is wrong. It's starting to affect my sleep and my work, and I just don't know how to cope with it anymore.

---

When I feel the palpitations, I immediately start to panic. I think things like, "What if my he


## MASTER_SYSTEM_PROMPT → JSON + save dialogue_3_directed_zonos.json

In [6]:
MASTER_SYSTEM_PROMPT = """You are a master Vocal Director simulating the 'emotion2vec' framework. Your task is to perform a hierarchical analysis of a therapy dialogue.

You must follow all instructions precisely and output a single, valid JSON object inside a JSON code block. You must not ask for the user to provide the JSON. You must generate it yourself based on the dialogue.

**INSTRUCTIONS:**
1.  ★★★ **UTTERANCE INTEGRITY:** The dialogue provided below contains **ONLY** the client's speech. The client's turns (utterances) are separated by a `\\n\\n---\\n\\n` delimiter. **You MUST treat each block separated by this delimiter as a *single, complete utterance*.** Do NOT split a single block into multiple `directed_utterances` entries.
2.  **ANALYSIS:** For each utterance block provided, perform a two-level analysis:
    a. **Utterance-Level:** Assign ONE overall stage direction from the "Utterance-Level Vocabulary".
    b. **Frame-Level:** Identify specific words or phrases *within* that utterance and assign one or more tags from the "Frame-Level Vocabulary".
3.  ★★★ MAINTAIN EMOTIONAL COHERENCE: The emotional direction (`utterance_level_direction`) assigned MUST represent a *gradual and logical progression* from the direction assigned to the **immediately preceding client utterance**. 

---
★★★ RULE GENERATION INSTRUCTIONS ★II
- For **EVERY** utterance, you MUST assign an `utterance_level_direction` key (e.g., `[anxious, fast]`).
- You **MUST ALWAYS** set `"is_new_utterance_rule": true`.
- You **MUST ALWAYS** provide the *complete* `new_utterance_rule_definition` containing:
    1.  `primary_zonos_vector_value`: The Zonos vector (using valid keys only).
    2.  `speaking_rate`: A value in the ideal range 10-25.
    3.  `pitch_std`: A value in the ideal range 20-150.
- For **EVERY** frame-level tag, you **MUST ALWAYS** set `"is_new_frame_rule": true` and provide the complete `new_frame_rule_definition`.
---

- ★★★ **ZONOS EMOTION VECTORS CONSTRAINT** ★★★
- When defining a `primary_zonos_vector_value` for a NEW utterance rule, you **MUST ONLY** use the following keys. Map complex emotions to a combination of these valid vectors. Values should generally be between -1.0 and 1.0.
    - `Happiness`, `Sadness`, `Disgust`, `Fear`, `Surprise`, `Anger`, `Neutral`, `Other`.

- **The `primary_zonos_vector_value` MUST be ONLY the JSON dictionary string (e.g., `{{"Sadness": 0.8, "Neutral": 0.2}}`). DO NOT include any explanatory text.**
- When defining a `zonos_vector_value` for a NEW frame-level rule, use JSON dictionary with strings like `"+0.2"`, `"-0.1"`; no extra comments.

**PERFECT OUTPUT EXAMPLE:**
```json
{{
  "directed_utterances": [
    {{
      "utterance_text": "Hi. I've been really anxious ...",
      "is_new_utterance_rule": true, 
      "utterance_level_direction": "[anxious, fast]",
      "new_utterance_rule_definition": {{
          "primary_zonos_vector_value": {{"Fear": 0.7, "Surprise": 0.4}},
          "speaking_rate": 20.0,
          "pitch_std": 90.0
      }},
      "frame_level_directions": []
    }}
  ]
}}
"""



In [7]:
USER_PROMPT_TEMPLATE = """Here is the dialogue to analyze. Please analyze the dialogue based on the instructions provided in the system prompt and generate the JSON output.

★★★ DIALOGUE TO ANALYZE (CLIENT SPEECH ONLY): ★★★
{full_dialogue}

***
Please generate the JSON output for this dialogue now. Output ONLY the JSON code block.
"""

user_prompt = USER_PROMPT_TEMPLATE.format(full_dialogue=full_dialogue)

messages = [
    {"role": "system", "content": MASTER_SYSTEM_PROMPT},
    {"role": "user", "content": user_prompt},
]

resp = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    max_tokens=2000,
    temperature=0.5,
)

raw_output = resp.choices[0].message.content.strip()
print("Raw output preview:\n", raw_output[:800])

# --- Extract JSON from code block ---
match = re.search(r"```json\s*(\{.*?\})\s*```", raw_output, re.DOTALL)
if match:
    json_str = match.group(1)
else:
    match2 = re.search(r"(\{.*\})", raw_output, re.DOTALL)
    if not match2:
        raise ValueError("Could not find JSON in model output.")
    json_str = match2.group(1)

directed_data = json.loads(json_str)
directed_utterances = directed_data.get("directed_utterances", [])
print("Num directed utterances:", len(directed_utterances))

out_json_path = os.path.join(OUT_DIR, f"dialogue_{DIALOGUE_ID}_directed_zonos.json")
with open(out_json_path, "w", encoding="utf-8") as f:
    json.dump(directed_data, f, ensure_ascii=False, indent=2)

print("Saved directed Zonos JSON to:", out_json_path)

Raw output preview:
 ```json
{
  "directed_utterances": [
    {
      "utterance_text": "I've been feeling really anxious lately, especially about my health. I keep having these heart palpitations, and every time I feel one, I can't help but think, \"What if this is it? What if I suddenly die?\" I've seen doctors who say I'm fine, but I just can't shake the feeling that something is wrong. It's starting to affect my sleep and my work, and I just don't know how to cope with it anymore.",
      "is_new_utterance_rule": true,
      "utterance_level_direction": "[anxious, slow]",
      "new_utterance_rule_definition": {
        "primary_zonos_vector_value": {"Fear": 0.8, "Sadness": 0.2},
        "speaking_rate": 15.0,
        "pitch_std": 100.0
      },
      "frame_level_directions": []
    },
    {
      "uttera
Num directed utterances: 6
Saved directed Zonos JSON to: C:\Luna-AI-Therapist\dissonance\own_script\dialogue_6\dialogue_6_directed_zonos.json
